# LogicCheck: un pequeño verificador de satisfacibilidad.
**Universidad Nacional de Colombia**
**Curso:** Introducción a las Ciencias de la Computación  
**Semestre:** 2026-1  
**Docente:** Carlos Manuel Orrego Franco  
**Grupo:** 7  
**Fecha de entrega:** 16 de junio de 2026  
**Integrantes:**
- Mariana Zapata Fernández
- Alaham Daniel Aarón Daza
- Andres Fabricio Pizo

---
## Resumen
Este informe desarrolla la evaluación de LogicCheck Lab, una herramienta educativa para mostrar cómo una computadora puede razonar sobre fórmulas proposicionales pequeñas. Se busca construir un verificador de satisfacibilidad proposicional (SAT) que determine si una fórmula lógica dada es satisfacible o no. Para esto se implementa un algoritmo de backtracking que explora el espacio de asignaciones posibles para las variables de la fórmula, aplicando técnicas de poda para descartar ramas del árbol de búsqueda que no pueden conducir a una solución satisfacible. El informe incluye una descripción detallada del procedimiento seguido, un modelamiento manual de un caso pequeño para ilustrar el funcionamiento del algoritmo, y una implementación en Python. Además, se discuten las decisiones tomadas durante el desarrollo, los resultados obtenidos y las conclusiones finales sobre la efectividad del enfoque utilizado.

## 1. Descripción del Problema
El problema fundamental que se plantea es el problema de Satisfacibilidad Booleana (SAT). El objetivo es determinar si existe una asignación de valores de verdad (True / False) a un conjunto de variables booleanas, de tal forma que una fórmula lógica completa se evalúe como verdadera. En nuestro contexto, modelaremos las fórmulas usando listas de cláusulas. Todas las cláusulas deben ser verdaderas para que el problema esté satisfecho, lo que convierte este desafío en una excelente aplicación para un algoritmo de Backtracking.

## 2. Modelamiento del Backtracking
Definimos los elementos clave de nuestro algoritmo según las reglas lógicas:

- **Estado (Solución parcial):** Un registro de los valores asignados a algunas variables hasta el momento. Usaremos un diccionario en Python (por ejemplo, `{"A": True, "B": None}`).
- **Candidatos:** Las opciones que nuestro código probará en cada paso recursivo serán estrictamente limitadas a valores booleanos `True` o `False` para la siguiente variable sin asignar.
- **Restricciones (Criterio de Poda):** Es la condición lógica que evalúa la viabilidad de la solución parcial. Una cláusula se define como *definitivamente falsa* si todos los literales que la componen han sido evaluados y el resultado de su disyunción es falso. Cuando se detecta una cláusula falsa antes de completar la asignación, se activa el mecanismo de poda.
- **Solución Completa:** Se alcanza cuando todas las variables tienen valor asignado y, al evaluar la fórmula, ninguna cláusula resulta ser falsa.
- **Retroceso (Backtrack):** Si al probar un candidato se genera una violación de las restricciones, el algoritmo descarta esa rama y retrocede para probar el siguiente candidato disponible.

## 3. Traza del Algoritmo Paso a Paso
Supongamos que tenemos la siguiente fórmula lógica:
`Fórmula = [["-A", "-B"], ["A", "B"]]`

El algoritmo inicializa con una asignación vacía `A={}` y procesa las variables en orden alfabético: `[A, B]`.

**Paso 1: Exploración de la primera rama (A=True)**
1. Asignación Parcial: `A={A: True}`
2. Evaluación de cláusulas:
   - Cláusula 1: `(-A ∨ -B)` se evalúa como `(-True ∨ -B) = (False ∨ -B)`. Para que esta cláusula no sea definitivamente falsa, B debe ser False.
   - Cláusula 2: `(A ∨ B)` se evalúa como `(True ∨ B) = True`, lo que es satisfactorio.

**Paso 2: Decisión sobre la siguiente variable (B=True)**
1. Asignación Parcial: `A={A: True, B: True}`
2. Evaluación de cláusulas:
   - Cláusula 1: `["-A", "-B"]`. Como A es True y B es True, la cláusula evalúa como `(False ∨ False) = False`.
3. Resultado: La cláusula 1 es definitivamente falsa. Se activa la **poda** y el algoritmo **retrocede** (backtrack) a la asignación anterior.

**Paso 3: Siguiente candidato (B=False)**
1. Asignación Parcial: `A={A: True, B: False}`
2. Evaluación de cláusulas:
   - Cláusula 1: `["-A", "-B"]` se evalúa como `(False ∨ True) = True`.
   - Cláusula 2: `["A", "B"]` ya era `True` por el paso 1.
3. Resultado: Todas las cláusulas de la fórmula son verdaderas y todas las variables han sido asignadas. El algoritmo se detiene con ÉXITO.

**Asignación Final Satisfacible:** `{A: True, B: False}`.

## 4. Implementación en Python
A continuación se presenta el código que aplica el backtracking para resolver fórmulas proposicionales. Incluye contadores de estadísticas para poder evaluar la eficiencia del algoritmo.

In [5]:
def get_variables(formula):
    """Extrae todas las variables únicas de una fórmula lógica."""
    vars_set = set()
    for clause in formula:
        for literal in clause:
            vars_set.add(literal.replace("-", ""))
    return sorted(list(vars_set))

def evaluate_clause(clause, assignment):
    """
    Evalúa si una cláusula es verdadera, falsa o incierta.
    Retorna True si es definitivamente verdadera,
    False si es definitivamente falsa,
    None si no se puede determinar aún.
    """
    is_false = True
    for literal in clause:
        is_neg = literal.startswith("-")
        var = literal.replace("-", "")
        val = assignment.get(var)

        if val is None:
            is_false = False  # Falta información, no es definitivamente falsa
        else:
            # Si un literal es True, toda la cláusula es True (disyunción)
            if (val and not is_neg) or (not val and is_neg):
                return True

    if is_false:
        return False  # Todos los literales evaluados fueron falsos

    return None

def solve_sat(formula, variables, index, assignment, stats):
    stats["llamadas_recursivas"] += 1

    # RESTRICCIONES (Poda)
    for clause in formula:
        if evaluate_clause(clause, assignment) is False:
            stats["ramas_podadas"] += 1
            return False

    # CASO BASE: Solución Completa
    if index == len(variables):
        return True

    var = variables[index]

    # CANDIDATOS
    for candidate in [True, False]:
        assignment[var] = candidate

        if solve_sat(formula, variables, index + 1, assignment, stats):
            return True

        # RETROCESO (Backtrack)
        stats["retrocesos"] += 1
        assignment[var] = None

    return False

def run_logic_check(formula):
    variables = get_variables(formula)
    assignment = {v: None for v in variables}
    stats = {"llamadas_recursivas": 0, "ramas_podadas": 0, "retrocesos": 0}

    is_satisfiable = solve_sat(formula, variables, 0, assignment, stats)

    if is_satisfiable:
        return True, {k: v for k, v in assignment.items() if v is not None}, stats
    else:
        return False, None, stats

## 5. Pruebas y Resultados
De acuerdo a los requisitos, hemos implementado al menos 3 casos de prueba satisfacibles y 2 insatisfacibles, reportando en cada uno si existe solución, la asignación encontrada y las estadísticas del algoritmo (llamadas, podas y retrocesos).

In [6]:
casos_prueba = [
    # 3 Fórmulas Satisfacibles
    ("Caso SAT 1 (Ejemplo Manual)", [["-A", "-B"], ["A", "B"]]),
    ("Caso SAT 2 (Implicación)", [["-A", "B"], ["A"]]),
    ("Caso SAT 3 (Múltiples Variables)", [["A", "B", "-C"], ["-A", "B"], ["C"]]),

    # 2 Fórmulas Insatisfacibles
    ("Caso UNSAT 1 (Contradicción Directa)", [["A"], ["-A"]]),
    ("Caso UNSAT 2 (Ciclo Imposible)", [["A", "B"], ["-A"], ["-B"]])
]

for nombre, formula in casos_prueba:
    print(f"--- {nombre} ---")
    print(f"Fórmula: {formula}")
    satisfiable, asignacion, stats = run_logic_check(formula)

    if satisfiable:
        print(f"Resultado: SATISFACIBLE \nAsignación: {asignacion}")
    else:
        print("Resultado: INSATISFACIBLE (No existe combinación válida)")

    print(f"Estadísticas: {stats}\n")

--- Caso SAT 1 (Ejemplo Manual) ---
Fórmula: [['-A', '-B'], ['A', 'B']]
Resultado: SATISFACIBLE 
Asignación: {'A': True, 'B': False}
Estadísticas: {'llamadas_recursivas': 4, 'ramas_podadas': 1, 'retrocesos': 1}

--- Caso SAT 2 (Implicación) ---
Fórmula: [['-A', 'B'], ['A']]
Resultado: SATISFACIBLE 
Asignación: {'A': True, 'B': True}
Estadísticas: {'llamadas_recursivas': 3, 'ramas_podadas': 0, 'retrocesos': 0}

--- Caso SAT 3 (Múltiples Variables) ---
Fórmula: [['A', 'B', '-C'], ['-A', 'B'], ['C']]
Resultado: SATISFACIBLE 
Asignación: {'A': True, 'B': True, 'C': True}
Estadísticas: {'llamadas_recursivas': 4, 'ramas_podadas': 0, 'retrocesos': 0}

--- Caso UNSAT 1 (Contradicción Directa) ---
Fórmula: [['A'], ['-A']]
Resultado: INSATISFACIBLE (No existe combinación válida)
Estadísticas: {'llamadas_recursivas': 3, 'ramas_podadas': 2, 'retrocesos': 2}

--- Caso UNSAT 2 (Ciclo Imposible) ---
Fórmula: [['A', 'B'], ['-A'], ['-B']]
Resultado: INSATISFACIBLE (No existe combinación válida)
Estadís

## 6. Caso de Estudio: El Enigma del Turno de Fin de Semana

**Historia (Descripción del Problema)**
Una empresa de software necesita organizar el turno de soporte técnico para este fin de semana. Tienen a tres programadores disponibles: **Ana (A)**, **Beto (B)** y **Carlos (C)**. Recursos Humanos ha establecido las siguientes reglas lógicas para asignar los turnos:
1. **Regla 1:** Al menos uno de los tres debe trabajar en el turno.
2. **Regla 2:** Ana y Beto no se llevan bien, por lo que *no pueden trabajar juntos*.
3. **Regla 3:** Carlos es un programador Junior y necesita supervisión. Si Carlos trabaja, *Ana debe trabajar obligatoriamente* para supervisarlo.
4. **Regla 4:** Por ser el líder del proyecto este mes, *Beto debe trabajar obligatoriamente*.

**Modelamiento y Representación de Datos**
Convertimos la historia a variables booleanas:
* $A$: Ana trabaja (`True` o `False`)
* $B$: Beto trabaja (`True` o `False`)
* $C$: Carlos trabaja (`True` o `False`)

Transformamos las reglas a formato de lista de cláusulas (donde cada lista interna es una disyunción "O" y todas deben cumplirse "Y"):
* Regla 1: `["A", "B", "C"]` *(A o B o C)*
* Regla 2: `["-A", "-B"]` *(No Ana o No Beto)*
* Regla 3: `["-C", "A"]` *(No Carlos o Ana)*
* Regla 4: `["B"]` *(Beto obligatoriamente)*

**Fórmula Final en el Código:**
`Fórmula = [["A", "B", "C"], ["-A", "-B"], ["-C", "A"], ["B"]]`

In [7]:
print("\n=== RESOLVIENDO: EL ENIGMA DEL TURNO DE FIN DE SEMANA ===")
formula_enigma = [["A", "B", "C"], ["-A", "-B"], ["-C", "A"], ["B"]]

# Ejecutamos el verificador usando la función que definimos en la sección 4
satisfiable, asignacion, stats = run_logic_check(formula_enigma)

if satisfiable:
    print(f"Resultado: SATISFACIBLE")
    print(f"Asignación encontrada: {asignacion}")
    print("\nConclusión del Caso:")
    print("El problema tiene una única solución lógica. Para cumplir todas las reglas de la empresa,")
    print("Beto es el único que debe trabajar este fin de semana. Ana no puede trabajar (porque no se lleva con Beto),")
    print("y Carlos tampoco puede trabajar (porque Ana no estará ahí para supervisarlo).")
else:
    print("Resultado: INSATISFACIBLE (Es una contradicción)")

print(f"\nEstadísticas del algoritmo: {stats}")


=== RESOLVIENDO: EL ENIGMA DEL TURNO DE FIN DE SEMANA ===
Resultado: SATISFACIBLE
Asignación encontrada: {'A': False, 'B': True, 'C': False}

Conclusión del Caso:
El problema tiene una única solución lógica. Para cumplir todas las reglas de la empresa,
Beto es el único que debe trabajar este fin de semana. Ana no puede trabajar (porque no se lleva con Beto),
y Carlos tampoco puede trabajar (porque Ana no estará ahí para supervisarlo).

Estadísticas del algoritmo: {'llamadas_recursivas': 8, 'ramas_podadas': 3, 'retrocesos': 4}


## 7. Decisiones del Grupo (Reflexión)

**1. ¿Qué lenguaje de programación usaron y por qué?**
Usamos Python, porque permite modelar rápidamente estructuras de datos como listas y diccionarios, y su legibilidad facilita implementar recursividad de forma intuitiva, lo que es ideal para Backtracking.

**2. ¿Qué representación eligieron y por qué?**
Representamos la fórmula como una lista de listas de cadenas (`[["A", "B"], ["-A"]]`), y la solución parcial como un diccionario (`{"A": True, "B": None}`). Esto nos permite evaluar rápida y fácilmente el estado de cualquier variable en $O(1)$.

**3. ¿Qué restricciones implementaron primero?**
La restricción crítica de que ninguna cláusula puede ser definitivamente falsa bajo la asignación parcial actual. Si una sola cláusula falla irremediablemente, la fórmula entera falla.

**4. ¿Qué bug encontraron durante el desarrollo?**
Inicialmente no distinguíamos entre una cláusula con valor "Falso definitivo" y una cláusula "Aún no evaluada completamente". Esto provocaba que podáramos ramas prematuramente. Lo solucionamos implementando un retorno tri-estado (`True`, `False`, `None`) en `evaluate_clause`.

**5. ¿Qué caso de prueba falló inicialmente?**
El caso de fórmulas insatisfacibles simples como `[["A"], ["-A"]]`. El algoritmo no lograba retroceder correctamente al principio porque las variables no se re-asignaban a `None` al realizar el Backtracking.

**6. ¿Qué poda implementaron?**
Podamos la rama en el mismo instante en que se detecta que cualquier cláusula de la fórmula se evalúa como *definitivamente falsa*. De esta forma, evitamos explorar el resto de variables que no cambiarían el hecho de que esa cláusula falló.
Se descarta inmediatamente cualquier candidato que viole las restricciones (una cláusula que no cuadra o un número repetido), ahorrando la exploración de millones de ramas inútiles.

**7. ¿Qué candidatos prueba el algoritmo?**
Para SAT: `True` y `False`.

**8. ¿Cuál fue la restricción más importante?**
En SAT, evitar que una cláusula se vuelva definitivamente falsa.

**9. ¿Dónde ocurre el retroceso en el código?**
Ocurre cuando la llamada recursiva falla y devuelve `False`. En SAT se hace `assignment[var] = None`

**10. ¿El algoritmo encuentra una, todas o la mejor solución?**
El algoritmo está diseñado para detenerse y retornar tan pronto como encuentra *la primera* asignación o tablero válido (una solución), lo cual es ideal para SAT.

**11. ¿Qué cambiarían si tuvieran más tiempo (o si el problema fuera más grande)?**
Implementaríamos una heurística de ordenamiento de variables para evaluar primero las variables que aparecen en más cláusulas o en cláusulas unitarias (como en el algoritmo DPLL), lo cual aumentaría la cantidad de podas tempranas y aceleraría el tiempo de ejecución.

**12. ¿Qué parte hizo cada integrante?**
- **Mariana:** Diseño general del algoritmo, modelamiento y visualización de la traza paso a paso.
- **Alaham:** Implementación de la función `evaluate_clause` y el chequeo de restricciones lógicas.
- **Andres:** Creación de los casos de prueba, ejecución de las estadísticas y ensamblado final del notebook.

## 8. Conclusiones
Se logró diseñar e implementar un verificador de satisfacibilidad proposicional utilizando el modelo de backtracking, lo que permitió resolver el problema de manera eficiente al evitar la generación de todas las combinaciones posibles. Observando las estadísticas en nuestras pruebas, es evidente cómo el mecanismo de retroceso y la evaluación temprana de restricciones descartan ramas inviables significativamente, previniendo así un decaimiento drástico del rendimiento (lo que ocurriría en un enfoque de fuerza bruta completa).